# Prediction Head Models

Imports and verifies P1 (classification-only), P2 (profile-only), and P3 (multi-task) prediction models.

In [1]:
import os
os.chdir(r'D:\424_project-main')  # ensure relative paths resolve from project root


In [2]:
import torch
from src.profile_models import P1_ClassificationOnly, P2_ProfileOnly, P3_MultiTaskProfile
from src.models import MultiTaskLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

rgb = torch.randn(2, 5, 3, 96, 96).to(device)
gps = torch.randn(2, 5, 9).to(device)
labels   = torch.randint(0, 256, (2,)).to(device)
profiles = torch.randn(2, 256).to(device)

loss_fn = MultiTaskLoss(lambda_prof=0.1, lambda_smooth=0.01, lambda_rank=0.05)

for name, ModelClass in [('P1', P1_ClassificationOnly),
                          ('P2', P2_ProfileOnly),
                          ('P3', P3_MultiTaskProfile)]:
    m = ModelClass().to(device)
    out = m(rgb, gps)
    loss, details = loss_fn(out, labels, profiles)
    n_params = sum(p.numel() for p in m.parameters()) / 1e6
    keys = list(out.keys())
    print(f'  {name}: outputs={keys}  loss={loss.item():.4f}  params={n_params:.2f}M')


Device: cuda


C:\Users\MITHIL\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


  P1: outputs=['logits']  loss=5.4957  params=16.85M
  P2: outputs=['pred_profile', 'logits']  loss=5.9846  params=16.85M


  P3: outputs=['logits', 'pred_profile', 'fused_rep']  loss=5.6541  params=17.11M
